In [1]:
import numpy as np
import pandas as pd

from astropy.io import fits

In [2]:
df6_one = fits.open("VII_259_6dfgs.dat.gz.fits")[1]
df6_one_columns = df6_one.columns
df6_one_data = df6_one.data
df6_one_columns

df6_1 = pd.DataFrame(df6_one_data)
df6_1.head()

,6dFGS,RAh,RAm,RAs,DE-,DEd,DEm,DEs,Nm,Nz,...,r_cz,q_cz,GLON,GLAT,AV,w,Target,Template,FileName,SpecID
0,g0000026-232205,0,0,2.58,-,23,22,4.9,1,1,...,126,4,48.04,-77.79,0.06,8,2,1,0007m230VRredz2,758
1,g0000033-360504,0,0,3.28,-,36,5,4.0,2,2,...,126,4,350.39,-75.88,0.05,8,3,3:3,2359m364VRredz2,917:117199
2,g0000042-522157,0,0,4.22,-,52,21,57.1,1,1,...,126,4,320.32,-62.99,0.05,8,4,2,0009m510VRredz2,1102
3,g0000043-501834,0,0,4.26,-,50,18,34.4,1,1,...,126,4,322.43,-64.83,0.05,8,5,2,0009m510VRredz2,1124
4,g0000044-000500,0,0,4.43,-,0,5,0.0,0,0,...,5,9,96.30,-60.27,0.11,8,6,Z,ZCAT,0


In [3]:
df6_two = np.loadtxt("df6_spectra.txt", delimiter="|", dtype="str", skiprows=6)
two_columns = "SpecID|Target|6dFGS|RA-DEC|dr|zraw|z|q_z|Rcor|ExpV|MJD.V|ExpR|MJD.R|Com".split("|")

df6_2 = pd.DataFrame(df6_two, columns=two_columns)
df6_2.head()

,SpecID,Target,6dFGS,RA-DEC,dr,zraw,z,q_z,Rcor,ExpV,MJD.V,ExpR,MJD.R,Com
0,1,104,g0000531-300714,00:00:53.07 -30:07:14.0,0.001,0.03975,0.03981,1,3.92,1200,52846.78,600,52846.82,
1,2,127050,g0001032-291829,00:01:03.23 -29:18:29.1,0.001,2.28448,2.28448,1,8.89,1200,52846.78,600,52846.82,
2,3,168322,g2359150-300636,23:59:14.98 -30:06:36.2,0.001,2.28377,2.28377,1,5.30,1200,52846.78,600,52846.82,
3,4,113826,g2358173-300641,23:58:17.26 -30:06:41.2,0.001,0.02961,0.02967,4,9.21,1200,52846.78,600,52846.82,
4,5,127047,g2358428-293647,23:58:42.75 -29:36:46.7,0.001,0.06070,0.06076,4,7.11,1200,52846.78,600,52846.82,


In [4]:
# SET THE COORDS

df6_1_ra = df6_1["RAh"] * 360/24 + df6_1["RAm"] * 360 / (24*60) + df6_1["RAs"] * 360 / (24*3600)
df6_1["RA"] = df6_1_ra

df6_1_dec_sign = 1 + (-2) * (df6_1["DE-"] == "-")
df6_1_dec = df6_1["DEd"] + df6_1["DEm"] / 60 + df6_1["DEs"] / 3600
df6_1["DEC"] = df6_1_dec_sign * df6_1_dec

In [5]:
df6_2["Target"] = np.int_(df6_2["Target"])
df6_2["z"] = np.float32(df6_2["z"])
df6_2["q_z"] = np.int_(df6_2["q_z"])

In [6]:
# CHOOSE RELEVANT LINES
df6_1_trunc = df6_1[["Target", "RA", "DEC", "GLON", "GLAT"]]
df6_1_trunc.head()

,Target,RA,DEC,GLON,GLAT
0,2,0.010750,-23.368028,48.04,-77.79
1,3,0.013667,-36.084444,350.39,-75.88
2,4,0.017583,-52.365861,320.32,-62.99
3,5,0.017750,-50.309556,322.43,-64.83
4,6,0.018458,-0.083333,96.30,-60.27


In [7]:
df6_2_trunc = df6_2[["Target", "RA-DEC", "z", "q_z"]]
df6_2_trunc.head()

,Target,RA-DEC,z,q_z
0,104,00:00:53.07 -30:07:14.0,0.03981,1
1,127050,00:01:03.23 -29:18:29.1,2.28448,1
2,168322,23:59:14.98 -30:06:36.2,2.28377,1
3,113826,23:58:17.26 -30:06:41.2,0.02967,4
4,127047,23:58:42.75 -29:36:46.7,0.06076,4


In [8]:
df6_trunc = pd.merge(df6_1_trunc, df6_2_trunc, how='inner', on="Target")
df6_trunc.head()

,Target,RA,DEC,GLON,GLAT,RA-DEC,z,q_z
0,2,0.010750,-23.368028,48.04,-77.79,00:00:02.58 -23:22:04.9,0.06660,4
1,3,0.013667,-36.084444,350.39,-75.88,00:00:03.29 -36:05:04.1,0.06044,4
2,3,0.013667,-36.084444,350.39,-75.88,00:00:03.29 -36:05:04.1,0.06038,4
3,4,0.017583,-52.365861,320.32,-62.99,00:00:04.22 -52:21:57.1,0.09181,4
4,5,0.017750,-50.309556,322.43,-64.83,00:00:04.26 -50:18:34.4,0.06617,4


In [9]:
galaxies = df6_trunc[df6_trunc['q_z'] > 2]
galaxies.size

939016

In [10]:
np.savez("6df-clean", ra=galaxies["RA"], dec=galaxies["DEC"], z=galaxies["z"])